In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import numpy as np

sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import read_KPI_adequacy, apply_conservative_classification, load_solutions, combine_solutions

G_save = False

# dim = (1000,500)
# g_BLUE = "1616A7"
# g_GREY = "#7F7F7F"
g_ORANGE = "Orange"



In [ ]:
def compute_ecdf(df: pd.DataFrame, value_col: str = "value", group_cols: list = None) -> pd.DataFrame:
    """
    Return a DataFrame with columns: group_cols..., value_col, ecdf
    where ecdf is the empirical CDF value in (0,1] for each sorted value within the group.
    """
    if group_cols is None:
        group_cols = []

    rows = []
    grouped = df.groupby(group_cols) if group_cols else [ (None, df) ]

    for name, g in grouped:
        vals = g[value_col].dropna().sort_values().reset_index(drop=True)
        n = len(vals)
        if n == 0:
            continue
        ecdf = (np.arange(n) + 1) / n
        tmp = pd.DataFrame({value_col: vals, "ecdf": ecdf})
        # attach group values
        if group_cols:
            if isinstance(name, tuple):
                for col, val in zip(group_cols, name):
                    tmp[col] = val
            else:
                tmp[group_cols[0]] = name
        rows.append(tmp)

    return pd.concat(rows, ignore_index=True)

In [ ]:

def get_ecdf_coordinates(day, model_type, variable, df, ecdf):
    value = df[(df['day'] == day) & (df['model_type'] == model_type) & (df['variable'] == variable)].value.values[0]
    filter = (ecdf['model_type'] == model_type) & (ecdf['variable'] == variable) & (ecdf['value'] == value)
    return ecdf[filter].value, ecdf[filter].ecdf

In [ ]:

# to_plot

In [ ]:
latex_textwidth_pt = 516.0 # double column # use this command in latex: \the\textwidth
# latex_textwidth_pt = 452.0 # single column
scale = 1
dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

def update_background(fig, legend_attr=legend_attr, dim=dim, showlegend=True):
    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)",
        # yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        # xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        width=dim[0],
        height=dim[1],
        showlegend=showlegend,
        autosize=False,
        legend=legend_attr,  # Include legend attributes
        # font=dict(
        #     family="Computer Modern",  # or 'Arial', 'Courier New', etc.
        #     # size=12,                   # default font size for all text
        # #     # color="black"              # font color
        #     )
    )
    config = dict(showgrid=False, showticklabels=True, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey")
    # fig.for_each_yaxis(lambda yaxis: yaxis.update(**config))
    # fig.for_each_xaxis(lambda xaxis: xaxis.update(**config))
    fig.update_yaxes(**config)
    
    fig.update_xaxes(**config)
    
    fig.update_traces(
        boxmean=True,
        selector=dict(type='box')
    )
    # for axis in fig.layout:
    #     if axis.startswith('xaxis') or axis.startswith('yaxis'):
    #         fig.layout[axis].update(showticklabels=True, linecolor="grey", mirror=True)

    # for axis in fig.layout:
    #     if axis.startswith('xaxis') or axis.startswith('yaxis'):
    #         fig.layout[axis].update(title = '')


In [ ]:

ss = [
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]

gcd_KPI_adequacy, gcdi_KPI_adequacy = read_KPI_adequacy(ss)
gcd_KPI_adequacy['E_ENS_%'] = gcd_KPI_adequacy['E_ENS_MWh']/gcd_KPI_adequacy['E_input_load_MWh']*100
gcd_KPI_adequacy['E_ENS_MWh'] = gcd_KPI_adequacy['E_ENS_MWh']*((gcd_KPI_adequacy['E_ENS_%'] >=0.01) + (gcd_KPI_adequacy['E_ENS_%'] <0.01))
gcd_KPI_adequacy['E_ENS_%']  = gcd_KPI_adequacy['E_ENS_%']*((gcd_KPI_adequacy['E_ENS_%'] >=0.01) + (gcd_KPI_adequacy['E_ENS_%'] <0.01))


Data construction and filtering

In [ ]:
filter_ = gcd_KPI_adequacy.pivot(
    index='day',
    columns = 'model_type',
    values='solution_id',
).dropna().index
gcd_KPI_adequacy = gcd_KPI_adequacy[gcd_KPI_adequacy['day'].isin(filter_)]
gcdi_KPI_adequacy = gcdi_KPI_adequacy[gcdi_KPI_adequacy['day'].isin(filter_)]


In [ ]:
cost_columns = ['OV_uc', 'thermal_production_cost_uc', 'E_OPEX', 'E_ENS_MWh', 'E_LLD_h']
# cost_differences = pd.DataFrame(index=filter_)
differences = gcd_KPI_adequacy.pivot(
    index='day',
    columns = 'model_type',
    values=cost_columns + ['E_input_load_MWh'],
    # var_name='cost_type',
    # value_name='cost_value'
)
for col in cost_columns:
    # differences[('LGEN_%', col)] = (differences[('LGEN_cost_uc', col)]) / differences[('OV_uc', col)] * 100
    differences[(col+'_diff_%','conservative')] = (differences[(col,'conservative')] - differences[(col,'e-reserve')]) / differences[(col,'e-reserve')]*100
    differences[(col+'_diff_%','envelope')] = (differences[(col,'envelope')] - differences[(col,'e-reserve')]) / differences[(col,'e-reserve')]*100
    differences[(col+'_diff','conservative')] = (differences[(col,'conservative')] - differences[(col,'e-reserve')])
    differences[(col+'_diff','envelope')] = (differences[(col,'envelope')] - differences[(col,'e-reserve')])
    # differences[(col+'_diff_%','conservative')] = (differences[(col,'conservative')] - differences[(col,'e-reserve')]) / differences[(col,'e-reserve')]*100
# diffe
differences = differences.stack(future_stack=True).dropna().reset_index()

differences['E_ENS_MWh_diff_%'] = differences['E_ENS_MWh_diff']/differences['E_input_load_MWh']*100
differences['E_LLD_h_diff_%'] = differences['E_LLD_h_diff']/24*100

In [ ]:
to_plot_name_map = {'OV_uc_diff_%': 'DA Cost Difference [%/day]',
                    # 'E_ENS_%_diff': 'RT Load Loss (%/day)',
                    # 'E_OPEX_diff_%': 'RT Cost Difference (%/day)',
                    # 'OPEX_uc': 'DA Cost ($/day)',
                    # 'E_OPEX': 'RT Cost ($/day)',
                    'E_ENS_MWh_diff_%': 'RT Unserved Energy [%/day]',
                    # 'E_ENS_%': 'RT Load Loss (%/day)',
                    # 'E_LLD_h': 'RT Load Loss Duration (h/day)',
                    }


left = gcd_KPI_adequacy.melt(
    id_vars=['day', 'model_type'],
    value_vars=['OPEX_uc','E_ENS_%', 'E_OPEX', 'E_LLD_h']

)
# left = pd.DataFrame()
to_plot = differences.melt(
    id_vars=['day', 'model_type'],
    value_vars=['OV_uc_diff_%',
                # 'E_ENS_MWh_diff_%',
                'E_ENS_MWh_diff_%',
                # 'E_LLD_h_diff_%'
                # 'E_LLD_h_diff'
                ],
    )

# to_plot = pd.concat([left, to_plot], axis=0)

# to_plot['variable'] = to_plot['variable'].replace(to_plot_name_map)
facet_mapping = {k:v for (k,v) in to_plot_name_map.items() if k in to_plot['variable'].unique()}
# to_plot = to_plot.sort_values('variable', key=lambda x: x.map(facet_mapping))
to_plot = to_plot.replace({'variable': facet_mapping})
to_plot = to_plot.replace({'model_type': {'envelope': 'dynamic'}})
# to_plot = to_plot.rename(columns={'model_type': 'model type'})

# to_plot = to_plot.replace({'OV_uc': 'DA system costs ($)'})

fig = px.box(
    to_plot,
    # x='model_type',
    y='value',
    color='model_type',
    facet_col='variable',
    category_orders={
        "model_type": ["conservative", "dynamic", "e-reserve", "stochastic"],
        "variable": list(facet_mapping.values())},
    labels = {'value': '', 'reserve_direction': '', 'model_type': 'model type'},
    # points="all",  # show all points
    # facet_order = list(to_plot_name_map.values()),
    # boxmode="group",
    facet_col_spacing=0.07,
    # hover_dat
)
update_background(fig,  legend_attr, dim)
# update_layout(fig, legend_attr=legend_attr, dim=dim, showlegend=True)
fig.update_yaxes(matches=None)

for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(title = '')

for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace(" ($/day)", "<br>[$/day]").replace(" [$/day]", "<br>[$/day]").replace(" [%/day]", "<br>[%/day]").replace(" [h/day]", "<br>[h/day]")
        annotation.text = annotation.text.replace("variable=", "")
        # annotation.text = ""

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()

In [ ]:
yearly_cost = gcd_KPI_adequacy.groupby('model_type')[['start_cost_uc', 'fixed_cost_uc', 'production_cost_uc', 'storage_production_cost_uc', 'LGEN_cost_uc', 'LOL_cost_uc']].sum()*(1e-6)
yearly_cost.round(2)

In [ ]:


ecdf = compute_ecdf(to_plot, value_col="value", group_cols=["model_type", "variable"])

# Create the ecdf plot
fig = px.ecdf(
    to_plot,
    x='value',
    color='model_type',
    hover_data='day',
    facet_col='variable',
    facet_col_spacing=0.07,
    labels = {'value': '', 'model_type': 'model type'},
)
# add_marker(fig)

marker= get_ecdf_coordinates(131, 'dynamic', 'RT Unserved Energy [%/day]', to_plot, ecdf)
fig.add_scatter(x = marker[0], y = marker[1], mode='markers', row = 1, col = 2, marker=dict(size=6, color = g_ORANGE, symbol="diamond"), showlegend=False)

marker= get_ecdf_coordinates(131, 'conservative', 'RT Unserved Energy [%/day]', to_plot, ecdf)
fig.add_scatter(x = marker[0], y = marker[1], mode='markers', row = 1, col = 2, marker=dict(size=6, color = g_ORANGE, symbol="diamond"), showlegend=False)

marker= get_ecdf_coordinates(131, 'dynamic', 'DA Cost Difference [%/day]', to_plot, ecdf)
fig.add_scatter(x = marker[0], y = marker[1], mode='markers', row = 1, col = 1, marker=dict(size=6, color = g_ORANGE, symbol="diamond"), showlegend=False)

marker= get_ecdf_coordinates(131, 'conservative', 'DA Cost Difference [%/day]', to_plot, ecdf)
fig.add_scatter(x = marker[0], y = marker[1], mode='markers', row = 1, col = 1, marker=dict(size=6, color = g_ORANGE, symbol="diamond"), showlegend=False)

fig.add_annotation(x=0.3, y=0.05, xref="paper", yref="paper", text="◆ day 131", showarrow=False, font=dict(color=g_ORANGE),)
fig.add_annotation(x=0.7, y=1-0.05, xref="paper", yref="paper", text="◆ day 131", showarrow=False, font=dict(color=g_ORANGE),)

update_background(fig, legend_attr, dim)
fig.update_xaxes(matches=None)
# update_background(fig, legend_attr, dim)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(title = '')

fig.update_layout(yaxis1=dict(title ='Cumulative probability')),
# fig.update_layout(xaxis1=dict(title ='%/day')),
# fig.update_layout(xaxis2=dict(title ='%/day')),

for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")
        # annotation.text = ""
n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()



In [ ]:
px.scatter(differences, x = 'OV_uc_diff_%', y = 'E_ENS_MWh_diff_%', color = 'model_type', hover_data = 'day')

In [ ]:
if G_save:
    fig.write_image("day_ahead_cost_&_unserved_energy.pdf", width=dim[0], height=dim[1])

In [ ]:
aux = gcd_KPI_adequacy[['model_type','OV_uc','E_ENS_MWh', 'OPEX_uc','E_ENS_%']].groupby('model_type').agg(['mean', 'std'])
print(aux)
print()

print("Relative cost reduction from e-reserve:", (aux.loc['conservative',('OV_uc','mean')] - aux.loc['e-reserve',('OV_uc','mean')]) / aux.loc['conservative',('OV_uc','mean')] *100)
print("Relative cost reduction from envelope:", (aux.loc['conservative',('OV_uc','mean')] - aux.loc['envelope',('OV_uc','mean')]) / aux.loc['conservative',('OV_uc','mean')] *100)
print("Absolute LOL_% increase to e-reserve:", (aux.loc['e-reserve',('E_ENS_%','mean')] - aux.loc['conservative',('E_ENS_%','mean')]))
print("Absolute LOL_% increase to envelope:", (aux.loc['envelope',('E_ENS_%','mean')] - aux.loc['conservative',('E_ENS_%','mean')]) )

In [ ]:
def add_up_down_multiindex(df):
    new_columns = []
    for col in df.columns:
        col_str = str(col)
        if 'up' in col_str:
            # Remove 'up_' or '_up' from the column name
            if 'up_' in col_str:
                base_col = col_str.replace('up_', '')
            else:
                base_col = col_str
            new_columns.append(('up', base_col))
        elif 'down' in col_str:
            # Remove 'down_' or '_down' from the column name
            if 'down_' in col_str:
                base_col = col_str.replace('down_', '')
            else:
                base_col = col_str
            new_columns.append(('down', base_col))
        else:
            new_columns.append(('', col))
    df.columns = pd.MultiIndex.from_tuples(new_columns)
    df = df.stack(level=0, future_stack=True)
    df.index = df.index.set_names('reserve_direction', level=-1)
    return df


reserve = gcd_KPI_adequacy.pivot(
    index=['day'],
    columns = 'model_type',
    values=['thermal_reserve_up_uc_MWh', 'thermal_reserve_down_uc_MWh', 'storage_reserve_up_uc_MWh', 'storage_reserve_down_uc_MWh',
            'thermal_energy_reserve_up_uc_MWh','thermal_energy_reserve_down_uc_MWh', 'storage_energy_reserve_up_uc_MWh', 'storage_energy_reserve_down_uc_MWh',
            'required_reserve_up_uc_MWh', 'required_reserve_down_uc_MWh','required_energy_reserve_up_uc_MWh', 'required_energy_reserve_down_uc_MWh',
            'E_storage_reserve_up_activation_MWh', 'E_storage_reserve_down_activation_MWh',
            'E_thermal_reserve_up_activation_MWh', 'E_thermal_reserve_down_activation_MWh'],
)

reserve = reserve.stack('model_type', future_stack = True).reset_index()
reserve.set_index(['day', 'model_type'], inplace=True)

reserve = add_up_down_multiindex(reserve)

In [ ]:
reserve['thermal_reserve_commitment'] = (
    reserve['thermal_reserve_uc_MWh'].fillna(0) + reserve['thermal_energy_reserve_uc_MWh'].fillna(0)
) / (
    reserve['required_reserve_uc_MWh'].fillna(0) +
    # reserve['storage_reserve_uc_MWh'].fillna(0) +
    reserve['required_energy_reserve_uc_MWh'].fillna(0) 
    # reserve['storage_energy_reserve_uc_MWh'].fillna(0)
)
reserve['storage_reserve_commitment'] = (
    reserve['storage_reserve_uc_MWh'].fillna(0) + reserve['storage_energy_reserve_uc_MWh'].fillna(0)
) / (
    reserve['required_reserve_uc_MWh'].fillna(0) +
    # reserve['storage_reserve_uc_MWh'].fillna(0) +
    reserve['required_energy_reserve_uc_MWh'].fillna(0) 
    # reserve['storage_energy_reserve_uc_MWh'].fillna(0)
)

# This KPI is the ratio of the energy activation to the total required reserve and energy reserve. Good for checking if the reserve is being activated as expected and is lower than 1. Not useful for energy reserves.
reserve['E_thermal_reserve_activation'] = reserve['E_thermal_reserve_activation_MWh'].fillna(0) / (reserve['required_reserve_uc_MWh'].fillna(0) + reserve['required_energy_reserve_uc_MWh'].fillna(0))

reserve['E_storage_reserve_activation'] = reserve['E_storage_reserve_activation_MWh'].fillna(0) / (reserve['required_reserve_uc_MWh'].fillna(0) + reserve['required_energy_reserve_uc_MWh'].fillna(0))


In [ ]:
to_plot  = reserve.reset_index().rename(columns={'storage_reserve_commitment': 'Storage Reserve Commitment', 'E_storage_reserve_activation': 'Storage Reserve Activation'})
to_plot['Batteries Reserve Commitment'] = to_plot['Storage Reserve Commitment']*100
to_plot['Storage Reserve Activation'] = to_plot['Storage Reserve Activation']*100
to_plot = to_plot.melt(
    id_vars=['day','model_type','reserve_direction'],
    value_vars=['Storage Reserve Commitment','Storage Reserve Activation'])
to_plot['value'] = to_plot['value'].clip(upper=100)
to_plot = to_plot.replace({'model_type': {'envelope': 'dynamic'}})
fig = px.box(
    to_plot,
    x='reserve_direction',
    y='value',
    color='model_type',
    facet_col='variable',
    category_orders={"model_type": ["conservative", "dynamic", "e-reserve", "stochastic"]},
    labels = {'value': 'Fraction of reserves', 'reserve_direction': '', 'model_type': 'model type'},
    boxmode="group",
)
update_background(fig,  legend_attr, dim)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
# fig.update_yaxes(matches=True)

for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        # annotation.text = annotation.text.replace(" ($/day)", "<br>($/day)").replace(" ($/day)", "<br>($/day)").replace(" (%/day)", "<br>(%/day)").replace(" (h/day)", "<br>(h/day)")
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()

In [ ]:
if G_save:
    fig.write_image("storage_reserve_participation.pdf", width=dim[0], height=dim[1])

In [ ]:
multiplier = pd.read_csv(os.path.join('..','input','RTS-GMLC_v2.4.2', 'uc', 'configuration_envelopes_e_reserve_mu_v3.csv'))
multiplier_avg = multiplier.groupby('day').mean().reset_index()
mu = multiplier.rename(columns={'mu_2_up': 'Up', 'mu_2_down': 'Down'})
mu = mu.melt(id_vars=['day','hour'], value_vars = ['Up', 'Down'])
median_mu = mu.groupby(['hour','variable']).median().reset_index()

In [ ]:
legend_attr = dict(
    x=0.5,
    y=-0.4,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig = px.box(mu, x ='hour', y = 'value', facet_col='variable',
             labels = {'value': '', 'variable': 'multiplier'}, hover_data = 'day')

fig.add_scatter(x = median_mu[median_mu.variable == 'Up'].hour,
              y = median_mu[median_mu.variable == 'Up'].value,
              mode='markers',
            #   line=dict(color=g_ORANGE, width=10),
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              # legend = False
)
fig.add_scatter(x = median_mu[median_mu.variable == 'Down'].hour,
              y = median_mu[median_mu.variable == 'Down'].value,
              mode='markers',
              col = 2, row = 1,
            #   line=dict(color=g_ORANGE, width=10),
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              # name = 'median'

)
update_background(fig,  legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)

fig.add_annotation(
  x=.59,
  y=.03,
  xref="paper",
  yref="paper",
  text="◆ median",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=.01,
  y=.03,
  xref="paper",
  yref="paper",
  text="◆ median",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
# fig.update_yaxes(matches=True)

for annotation in fig.layout.annotations:
    if annotation.text.startswith("multiplier="):
        # annotation.text = annotation.text.replace(" ($/day)", "<br>($/day)").replace(" ($/day)", "<br>($/day)").replace(" (%/day)", "<br>(%/day)").replace(" (h/day)", "<br>(h/day)")
        annotation.text = annotation.text.replace("multiplier=", "")
    

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
fig.show()

In [ ]:
if G_save:
    fig.write_image("multipliers_boxplot.pdf", width=dim[0], height=dim[1])

In [ ]:
aux = multiplier.melt(id_vars = ['hour','day'], value_vars = ['mu_2_up', 'mu_2_down'])
px.line(
    aux, x='hour',y = 'value', line_dash = 'variable', color = 'day'
)


In [ ]:



days = range(1,366)
s_uc = []
s_ed = []
solution_keys = ['reserve','energy_reserve']

for sol in ss:
    s = sol['solution_folder']
    s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    else:
        s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)


# for k,v in s_uc.items():
#     if 'µ' in v.columns:
#         s_uc[k] = apply_conservative_classification(s_uc[k])
# for k,v in s_ed.items():
#     if 'µ' in v.columns:
#         s_ed[k]= apply_conservative_classification(s_ed[k])



In [ ]:
s_battery = s_uc['energy_reserve'][s_uc['energy_reserve'].resource == 'battery']
s_battery =  s_battery[s_battery.model_type == 'e-reserve']

s_system = s_uc['energy_reserve'][s_uc['energy_reserve'].resource == 'system']
s_system =  s_system[s_system.model_type == 'e-reserve']

In [ ]:
# Following groups all r_id together
s_battery_sum = s_battery.groupby(['day','hour_i', 'hour'])[['energy_reserve_up_MW', 'energy_reserve_down_MW']].sum().rename(columns={'energy_reserve_up_MW': 'Up', 'energy_reserve_down_MW': 'Down'})
s_system_sum = s_system.groupby(['day','hour_i', 'hour'])[['required_energy_reserve_up_MW', 'required_energy_reserve_down_MW']].sum().rename(columns={'required_energy_reserve_up_MW': 'Up', 'required_energy_reserve_down_MW': 'Down'})
fraction = (s_battery_sum / s_system_sum).reset_index()
fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['duration'])[['Up','Down']].mean().reset_index()


In [ ]:



# to_plot.rename(columns={'variable': {'energy_reserve_up_MW': 'test'}}, inplace=True)
fig= px.box(
    fraction.melt(id_vars=['duration'], value_vars=['Up', 'Down']),
    x = 'duration',
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of energy <br> reserves  by batteries', 'duration': 'Duration [h]'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean['duration'],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean['duration'],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=1-.59,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
# fig.show()

In [ ]:

if G_save:
    fig.write_image("energy_reserves_by_batteries.pdf", width=dim[0], height=dim[1], scale=1)

In [ ]:
s_battery = s_uc['reserve'][s_uc['reserve'].resource.isin(['battery'])] #'hydro_reservoir', 'CSP'
s_battery =  s_battery[(s_battery.model_type == 'envelope') & (s_battery['µ'] == 'mu_2')]

s_system = s_uc['reserve'][s_uc['reserve'].resource == 'system']
s_system =  s_system[(s_system.model_type == 'envelope') & (s_system['µ'] == 'mu_2')]

In [ ]:
s_battery_sum = s_battery.groupby(['day','hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_system_sum = s_system.groupby(['day','hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
fraction = (s_battery_sum / s_system_sum).reset_index()
# fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['hour'])[['Up','Down']].mean().reset_index()

In [ ]:
x_axis = 'hour'
fig= px.box(
    fraction.melt(id_vars=[x_axis], value_vars=['Up', 'Down']),
    x = x_axis,
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of reserves by batteries'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=0.03,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
# fig.show()

In [ ]:

if G_save:
    fig.write_image("reserves_by_batteries.pdf", width=dim[0], height=dim[1], scale=1)

In [ ]:
tuples = [(d, h_i, h) for d in s_battery.day.unique() for h in s_battery.hour.unique() for h_i in s_battery.hour.unique() if h_i <= h]
s_battery_energy = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
s_battery_energy = s_battery_energy.merge(s_battery, left_on=['day','hour'], right_on=['day','hour']) 
s_battery_energy.set_index(['day','hour_i','hour'], inplace=True)
# Following groups all batteries together
s_battery_energy_sum = s_battery_energy.groupby(['day','hour_i', 'hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_battery_energy_sum = s_battery_energy_sum.groupby(['day','hour_i']).cumsum()
# s_battery_energy_sum

s_system_energy = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
s_system_energy = s_system_energy.merge(s_system, left_on=['day','hour'], right_on=['day','hour']) 
s_system_energy.set_index(['day','hour_i','hour'], inplace=True)
# Following groups all together (not needed in theory because there is just one system)
s_system_energy_sum = s_system_energy.groupby(['day','hour_i', 'hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
s_system_energy_sum = s_system_energy_sum.groupby(['day','hour_i']).cumsum()
# s_system_energy_sum

In [ ]:
fraction = (s_battery_energy_sum / s_system_energy_sum).reset_index()
fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['duration'])[['Up','Down']].mean().reset_index()
# mean = fraction.groupby(['hour_i'])[['Up','Down']].mean().reset_index()

In [ ]:
# x_axis = 'hour_i'
x_axis = 'duration'
# x_axis = 'hour'
fig= px.box(
    fraction.melt(id_vars=[x_axis], value_vars=['Up', 'Down']),
    x = x_axis,
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of reserves by batteries'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=1-.59,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
# fig.show()

In [ ]:
required_reserve = s_uc['reserve'][['hour','day','resource','µ','required_reserve_up_MW', 'required_reserve_down_MW']]
required_reserve = required_reserve[(required_reserve.resource == 'system')&(required_reserve['µ'] == 'mu_1')]
required_reserve.set_index(['day','hour'], inplace=True)
cum_required_reserve = required_reserve.groupby('day')[['required_reserve_up_MW', 'required_reserve_down_MW']].cumsum()
cum_required_reserve.rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'}, inplace=True)
cum_required_reserve['type'] = 'cumulative_reserve'

required_e_reserve = s_uc['energy_reserve'][['hour','hour_i','day','resource','required_energy_reserve_up_MW', 'required_energy_reserve_down_MW']]
required_e_reserve = required_e_reserve[(required_e_reserve.resource == 'system') & (required_e_reserve.hour_i == 1)]
required_e_reserve.set_index(['day','hour'], inplace=True)
required_e_reserve.rename(columns={'required_energy_reserve_up_MW': 'Up', 'required_energy_reserve_down_MW': 'Down'}, inplace=True)
required_e_reserve['type']= 'energy_reserve'

required_all = pd.concat([pd.melt(required_e_reserve.reset_index(), id_vars=['hour','day','type'], value_vars=['Up', 'Down']),
                pd.melt(cum_required_reserve.reset_index(), id_vars=['hour','day','type'], value_vars=['Up', 'Down'])])
required_day = required_all[required_all.day == 131]

In [ ]:
required_day = required_day.pivot(
    index='hour',
    columns=['type','variable'],
    values='value'
    )

In [ ]:
fig = px.box(required_all, x = 'hour', y = 'value', color = 'type', facet_col='variable', hover_data='day',boxmode="group", labels = {'value': 'energy requirements [MWh]'})



# fig.add_scatter(
#               y = required_day[('cumulative_reserve','Up')],
#               x = required_day.index,
#               mode='markers',
#               marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
#               col = 1, row = 1,
# )
# fig.add_scatter(
#               y = required_day[('energy_reserve','Up')],
#               x = required_day.index,
#               mode='markers',
#               marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
#               col = 1, row = 1,
# )
# fig.add_scatter(
#               y = required_day[('cumulative_reserve','Down')],
#               x = required_day.index,
#               mode='markers',
#               marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
#               col = 2, row = 1,
# )
# fig.add_scatter(
#               y = required_day[('energy_reserve','Down')],
#               x = required_day.index,
#               mode='markers',
#               marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
#               col = 2, row = 1,
# )

# fig.add_annotation(
#   x=1-.59,
#   y=1-.03,
#   xref="paper",
#   yref="paper",
#   text="◆ day 131",
#   showarrow=False,
#   font=dict(color=g_ORANGE),
# )
# fig.add_annotation(
#   x=1-.01,
#   y=1-.03,
#   xref="paper",
#   yref="paper",
#   text="◆ day 131",
#   showarrow=False,
#   font=dict(color=g_ORANGE),
# )


update_background(fig, legend_attr, dim, True)

fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))

In [ ]:
G_save = True
if G_save:
    fig.write_image("cumulative_energy_requirements.png", width=dim[0], height=dim[1], scale=1)

In [ ]:
day = 131
fig = px.line(required_all[required_all.day == day].sort_values('hour'), x='hour', y='value', color='type', facet_col='variable', symbol='type', labels = {'value': 'energy requirements [MWh]'})
update_background(fig, legend_attr, dim, True)

fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))

In [ ]:

if G_save:
    fig.write_image(f"cumulative_energy_requirements_day_{day}.png", width=dim[0], height=dim[1], scale=1)